# Auction Outcome Feasibility: Supervised Learning with an SVM

**CIE 3 demonstration notebook**

This notebook demonstrates two syllabus concepts using the UCI *Auction Verification* dataset:

1. **Supervised learning**: learn from labelled records by separating inputs (`X`) from the target (`y`), training on one partition and evaluating on a held-out partition.
2. **Support Vector Machine (SVM)**: fit one SVM classification boundary and use it to predict whether an unseen **Model Outcome** is feasible.

**What the prediction means.** An *Outcome Feasibility Prediction* estimates whether a price (and, optionally, a winner) is **possible within the modeled simultaneous multi-round auction**, approximating the dataset's formal model-checking result (`verification.result`).
It is **not** fraud detection, authenticity checking, a fairness assessment, or a statement about legal validity.

*Runtime:* Google Colab, CPU only. Use **Runtime → Run all**.

## 1. Dependencies

Google Colab already includes every library this notebook needs (pandas, scikit-learn, matplotlib, joblib), so nothing is installed. The scikit-learn version is printed and saved in the metadata, so the Streamlit demonstration can load the model with a matching version.

In [ ]:
import hashlib
import io
import json
import pathlib
import platform
import urllib.request
import zipfile

import joblib
import matplotlib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)
from sklearn.compose import ColumnTransformer
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    confusion_matrix,
    f1_score,
    make_scorer,
    matthews_corrcoef,
    precision_score,
    recall_score,
)
from sklearn.model_selection import (
    GridSearchCV,
    RepeatedStratifiedKFold,
    StratifiedKFold,
    cross_validate,
    train_test_split,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

print("Python       :", platform.python_version())
print("pandas       :", pd.__version__)
print("scikit-learn :", sklearn.__version__)
print("matplotlib   :", matplotlib.__version__)
print("joblib       :", joblib.__version__)


## 2. Load the official dataset

The data comes straight from the UCI Machine Learning Repository archive. A SHA-256 hash of `data.csv` is recorded so the exact data behind the model can be identified later.

> Ordoni, E., Bach, J., Fleck, A., & Bach, J. (2022). *Auction Verification* [Dataset]. UCI Machine Learning Repository. https://doi.org/10.24432/C52K6N. Licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

In [ ]:
DATASET_URL = "https://archive.ics.uci.edu/static/public/713/auction+verification.zip"
DATASET_MEMBER = "data.csv"
# SHA-256 of data.csv in the official archive when this notebook was written (informational check).
REFERENCE_DATASET_SHA256 = "031ed3ef598a4fb3c7a635ae8041371459bda56ac45dfa3d7b27def76d7e1254"

with urllib.request.urlopen(DATASET_URL, timeout=60) as response:
    archive_bytes = response.read()

with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    csv_bytes = archive.read(DATASET_MEMBER)

DATASET_SHA256 = hashlib.sha256(csv_bytes).hexdigest()
df = pd.read_csv(io.BytesIO(csv_bytes))

print("Dataset SHA-256:", DATASET_SHA256)
print("Matches reference hash:", DATASET_SHA256 == REFERENCE_DATASET_SHA256)
print("Shape:", df.shape)
df.head()

## 3. The seven inputs and the binary target

Each row is one **Auction Configuration** that was checked by a formal model checker.

| Column | Role | Meaning |
|---|---|---|
| `process.b1.capacity` | Input (numeric) | Capacity of bidder 1 (how many products it can win) |
| `process.b2.capacity` | Input (numeric) | Capacity of bidder 2 |
| `process.b3.capacity` | Input (numeric) | Capacity of bidder 3 |
| `process.b4.capacity` | Input (numeric) | Capacity of bidder 4 |
| `property.price` | Input (numeric) | Price being checked for the product |
| `property.product` | Input (categorical code) | Which product (1–6) the price refers to |
| `property.winner` | Input (categorical code) | Bidder being checked as winner (1–4); `0` means no winner specified |
| `verification.result` | **Target** | `True` = the outcome is **feasible** in the modeled auction; `False` = infeasible |
| `verification.time` | **Excluded** | Model-checker runtime. It is recorded only *after* verification runs, so it is never used as an input |

Product and winner are **codes, not quantities** (product 4 is not "twice" product 2), so they are one-hot encoded instead of scaled.

## 4. Validate the dataset contract

Lightweight checks before any learning happens: required columns, expected row count, no missing values, and a binary target.

In [ ]:
NUMERIC_FEATURES = [
    "process.b1.capacity",
    "process.b2.capacity",
    "process.b3.capacity",
    "process.b4.capacity",
    "property.price",
]
CATEGORICAL_FEATURES = ["property.product", "property.winner"]
FEATURES = NUMERIC_FEATURES + CATEGORICAL_FEATURES
TARGET = "verification.result"
EXCLUDED = "verification.time"
EXPECTED_ROWS = 2043

missing_columns = set(FEATURES + [TARGET]) - set(df.columns)
assert not missing_columns, f"Missing required columns: {missing_columns}"
assert len(df) == EXPECTED_ROWS, f"Expected {EXPECTED_ROWS} rows, found {len(df)}"
assert df[FEATURES + [TARGET]].notna().all().all(), "Missing values found in inputs or target"
assert set(df[TARGET].unique()) == {True, False}, f"Target is not binary: {df[TARGET].unique()}"

print("✓ All required columns present")
print(f"✓ {len(df)} records")
print("✓ No missing input or target values")
print("✓ Target is binary:", sorted(df[TARGET].unique()))

## 5. Target-class distribution

Feasible outcomes are the minority class, so the target is **imbalanced**. This affects both how the SVM is trained (class weighting) and how it is evaluated (MCC rather than accuracy alone).

In [ ]:
class_counts = df[TARGET].value_counts().rename(index={False: "Infeasible (False)", True: "Feasible (True)"})
distribution = pd.DataFrame({"count": class_counts, "share": (class_counts / len(df)).round(3)})
display(distribution)

ax = class_counts.plot.bar(rot=0, color=["tab:gray", "tab:blue"], figsize=(5, 3))
ax.set_ylabel("Records")
ax.set_title("verification.result")
plt.tight_layout()
plt.show()

## 6. Separate inputs (X) and target (y)

This is the core of supervised learning: the model sees `X` and learns to reproduce the labels in `y`. The target is encoded as `1` = feasible and `0` = infeasible.

In [ ]:
X = df[FEATURES].copy()
y = df[TARGET].astype(int)  # 1 = feasible, 0 = infeasible

assert EXCLUDED not in X.columns, "verification.time must never be a model input"
assert list(X.columns) == FEATURES

print("X shape:", X.shape)
print("y shape:", y.shape)
print("✓ verification.time is absent from X")

## 7. Fixed stratified 80/20 train/test split

* **80 %** of records are used for training; **20 %** are held out and never seen during training.
* **Stratified** keeps the same feasible/infeasible ratio in both partitions.
* `random_state=42` makes the split identical every time the notebook runs.

In [ ]:
RANDOM_STATE = 42
TEST_SIZE = 0.2

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
)

split_summary = pd.DataFrame(
    {
        "records": [len(y_train), len(y_test)],
        "feasible": [int(y_train.sum()), int(y_test.sum())],
        "feasible share": [round(y_train.mean(), 3), round(y_test.mean(), 3)],
    },
    index=["train", "test"],
)
split_summary

## 8. Leakage-safe pipeline and bounded hyperparameter search

Preprocessing and the SVM remain wrapped in one scikit-learn `Pipeline`:

* `StandardScaler` puts the numeric inputs on a common scale.
* `OneHotEncoder(handle_unknown="ignore")` turns product and winner codes into indicator columns.
* Every cross-validation fold fits its own preprocessing steps, preventing validation-fold leakage.

The search uses **only the 80% training partition**. It evaluates 16 combinations of `C` and `gamma` with five-fold stratified cross-validation (80 fits), selecting the configuration with the highest mean MCC. The untouched 20% test partition is not used for model selection.

`class_weight="balanced"` remains fixed because feasible outcomes are the minority class. `probability=False` is retained because the dashboard reports the SVM decision margin, not a calibrated probability.


In [ ]:
BASE_SVM_CONFIG = {
    "kernel": "rbf",
    "class_weight": "balanced",
    "probability": False,
}

PARAM_GRID = {
    "svm__C": [0.1, 1.0, 10.0, 100.0],
    "svm__gamma": ["scale", 0.01, 0.1, 1.0],
}

preprocess = ColumnTransformer(
    transformers=[
        ("numeric", StandardScaler(), NUMERIC_FEATURES),
        ("categorical", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ]
)

model_template = Pipeline(
    steps=[
        ("preprocess", preprocess),
        ("svm", SVC(**BASE_SVM_CONFIG)),
    ]
)

print("Parameter combinations:", np.prod([len(values) for values in PARAM_GRID.values()]))
model_template


## 9. Select the SVM configuration and measure stability

The first stage chooses `C` and `gamma` using five-fold stratified cross-validation on the training partition. The best pipeline is then refitted automatically on all training records.

The second stage evaluates that selected configuration with five-fold stratified cross-validation repeated five times (25 fits). This reports mean and standard deviation for MCC, accuracy, feasible precision, feasible recall, and feasible F1.

The repeated result is a **stability estimate for the selected configuration**, not a fully nested estimate of the selection procedure. The untouched test partition remains the final independent evaluation.


In [ ]:
MCC_SCORER = make_scorer(matthews_corrcoef)
SEARCH_CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

search = GridSearchCV(
    estimator=model_template,
    param_grid=PARAM_GRID,
    scoring=MCC_SCORER,
    cv=SEARCH_CV,
    n_jobs=-1,
    refit=True,
    return_train_score=False,
    error_score="raise",
)
search.fit(X_train, y_train)

model = search.best_estimator_
selected_svm = model.named_steps["svm"]
SVM_CONFIG = {
    **BASE_SVM_CONFIG,
    "C": float(selected_svm.C),
    "gamma": selected_svm.gamma,
}

selection_results = (
    pd.DataFrame(search.cv_results_)
    .loc[:, ["param_svm__C", "param_svm__gamma", "mean_test_score", "std_test_score", "rank_test_score"]]
    .sort_values(["rank_test_score", "std_test_score"])
    .reset_index(drop=True)
)
selection_results.columns = ["C", "gamma", "mean_cv_mcc", "std_cv_mcc", "rank"]

print("Selected parameters :", search.best_params_)
print("Best five-fold MCC  :", round(float(search.best_score_), 4))
print("Training records    :", len(X_train))
print(
    "Support vectors/class:",
    dict(zip(["infeasible (0)", "feasible (1)"], selected_svm.n_support_.tolist())),
)
display(selection_results.head(10))

REPEATED_CV_SPLITS = 5
REPEATED_CV_REPEATS = 5
REPEATED_CV = RepeatedStratifiedKFold(
    n_splits=REPEATED_CV_SPLITS,
    n_repeats=REPEATED_CV_REPEATS,
    random_state=RANDOM_STATE,
)
CV_SCORING = {
    "mcc": MCC_SCORER,
    "accuracy": make_scorer(accuracy_score),
    "precision_feasible": make_scorer(precision_score, pos_label=1, zero_division=0),
    "recall_feasible": make_scorer(recall_score, pos_label=1, zero_division=0),
    "f1_feasible": make_scorer(f1_score, pos_label=1, zero_division=0),
}

cv_raw = cross_validate(
    model,
    X_train,
    y_train,
    scoring=CV_SCORING,
    cv=REPEATED_CV,
    n_jobs=-1,
    return_train_score=False,
    error_score="raise",
)

cv_summary = {
    metric: {
        "mean": round(float(np.mean(cv_raw[f"test_{metric}"])), 4),
        "std": round(float(np.std(cv_raw[f"test_{metric}"], ddof=1)), 4),
    }
    for metric in CV_SCORING
}
cv_summary_frame = pd.DataFrame(cv_summary).T
cv_summary_frame.index.name = "metric"
display(cv_summary_frame)


## 10. Final evaluation on the untouched test partition

The selected pipeline is evaluated once on the 20% test partition that was excluded from parameter selection and repeated cross-validation.

**Matthews Correlation Coefficient (MCC)** remains the primary metric. It ranges from −1 to +1 and accounts for all four confusion-matrix cells.

**Why accuracy alone is not enough:** most records are infeasible, so a model that always predicted "infeasible" would still reach high accuracy while never finding a feasible outcome. That baseline's accuracy is printed for reference; its MCC is 0.


In [ ]:
y_pred = model.predict(X_test)

metrics = {
    "mcc": matthews_corrcoef(y_test, y_pred),
    "accuracy": accuracy_score(y_test, y_pred),
    "precision_feasible": precision_score(y_test, y_pred, pos_label=1, zero_division=0),
    "recall_feasible": recall_score(y_test, y_pred, pos_label=1, zero_division=0),
    "f1_feasible": f1_score(y_test, y_pred, pos_label=1, zero_division=0),
}
metrics = {name: round(float(value), 4) for name, value in metrics.items()}

cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
always_infeasible_accuracy = round(float((y_test == 0).mean()), 4)

display(pd.Series(metrics, name="held-out test").to_frame())
print("Accuracy of always predicting 'infeasible':", always_infeasible_accuracy)

ARTIFACT_DIR = pathlib.Path("artifacts")
ARTIFACT_DIR.mkdir(exist_ok=True)
CONFUSION_PNG = ARTIFACT_DIR / "confusion_matrix.png"

fig, ax = plt.subplots(figsize=(4.5, 4))
ConfusionMatrixDisplay(cm, display_labels=["Infeasible (0)", "Feasible (1)"]).plot(
    ax=ax, cmap="Blues", colorbar=False, values_format="d"
)
ax.set_title("Held-out test confusion matrix")
plt.tight_layout()
fig.savefig(CONFUSION_PNG, dpi=150)
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f"True negatives : {tn}   (infeasible, correctly rejected)")
print(f"False positives: {fp}   (infeasible, wrongly predicted feasible)")
print(f"False negatives: {fn}   (feasible, missed)")
print(f"True positives : {tp}   (feasible, correctly found)")

**Interpreting the result.** The repeated cross-validation summary describes the selected model's stability within the training partition. The final numbers above are performance on an untouched test partition drawn from the same dataset distribution.

Neither result shows that the model generalises to other auction designs, products, or bidder capacities outside the observed ranges.


## 11. Predict one unseen configuration

The record below comes from the held-out test partition, so the SVM never saw it during training. It is supplied as a plain seven-field record, the same way the Streamlit demonstration will send it.

In [ ]:
unseen_index = X_test.index[0]
unseen_record = {feature: int(X_test.loc[unseen_index, feature]) for feature in FEATURES}
unseen_frame = pd.DataFrame([unseen_record], columns=FEATURES)

prediction = model.predict(unseen_frame)
repeat_prediction = model.predict(unseen_frame)

# Lightweight checks: one binary prediction, deterministic on repetition.
assert prediction.shape == (1,), f"Expected exactly one prediction, got shape {prediction.shape}"
assert int(prediction[0]) in (0, 1), f"Prediction is not binary: {prediction[0]}"
assert np.array_equal(prediction, repeat_prediction), "Repeated prediction changed"
assert np.array_equal(model.predict(X_test), y_pred), "Repeated test-set prediction changed"

CLASS_LABELS = {0: "INFEASIBLE", 1: "FEASIBLE"}
print("Auction Configuration:")
for feature, value in unseen_record.items():
    print(f"  {feature:22s} = {value}")
print()
print("SVM prediction     :", CLASS_LABELS[int(prediction[0])])
print("Recorded label     :", CLASS_LABELS[int(y_test.loc[unseen_index])])
print("✓ Exactly one binary prediction; repeated predictions are identical")

## 12. Export the fitted pipeline and metadata

The bundle used by the Streamlit application contains:

* `auction_model.joblib`: the selected fitted pipeline (preprocessing + RBF SVM)
* `model_metadata.json`: features, class meanings, domains, split, selected SVM configuration, versions, and hashes
* `metrics.json`: model-selection details, repeated cross-validation mean ± standard deviation, untouched-test metrics, baseline, and confusion matrix
* `confusion_matrix.png`: the untouched-test plot above


In [ ]:
MODEL_PATH = ARTIFACT_DIR / "auction_model.joblib"
METADATA_PATH = ARTIFACT_DIR / "model_metadata.json"
METRICS_PATH = ARTIFACT_DIR / "metrics.json"
ZIP_PATH = pathlib.Path("auction_model_bundle.zip")


def sha256_of(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


joblib.dump(model, MODEL_PATH)
MODEL_SHA256 = sha256_of(MODEL_PATH)

input_domains = {
    feature: {
        "type": "numeric" if feature in NUMERIC_FEATURES else "categorical",
        "min": int(df[feature].min()),
        "max": int(df[feature].max()),
        "observed_values": sorted(int(v) for v in df[feature].unique()),
    }
    for feature in FEATURES
}

model_selection_report = {
    "method": "GridSearchCV",
    "primary_metric": "mcc",
    "training_partition_only": True,
    "cv": {
        "method": "StratifiedKFold",
        "splits": SEARCH_CV.n_splits,
        "shuffle": True,
        "random_state": RANDOM_STATE,
    },
    "parameter_grid": PARAM_GRID,
    "candidate_count": len(search.cv_results_["params"]),
    "best_params": {
        "C": float(selected_svm.C),
        "gamma": selected_svm.gamma,
    },
    "best_mean_cv_mcc": round(float(search.best_score_), 4),
}

repeated_cv_report = {
    "scope": "selected configuration evaluated on the training partition only",
    "method": "RepeatedStratifiedKFold",
    "splits": REPEATED_CV_SPLITS,
    "repeats": REPEATED_CV_REPEATS,
    "total_fits": REPEATED_CV_SPLITS * REPEATED_CV_REPEATS,
    "shuffle": True,
    "random_state": RANDOM_STATE,
    "metrics": cv_summary,
}

metrics_report = {
    "evaluation": "untouched held-out stratified test partition",
    "primary_metric": "mcc",
    "positive_class": 1,
    "model_selection": model_selection_report,
    "repeated_cross_validation": repeated_cv_report,
    "metrics": metrics,
    "confusion_matrix": {
        "labels": [0, 1],
        "matrix": cm.tolist(),
        "true_negative": int(tn),
        "false_positive": int(fp),
        "false_negative": int(fn),
        "true_positive": int(tp),
    },
    "test_records": int(len(y_test)),
    "test_feasible_records": int(y_test.sum()),
    "always_infeasible_accuracy": always_infeasible_accuracy,
}

metadata = {
    "model": "Outcome Feasibility Prediction (scikit-learn Pipeline: ColumnTransformer + SVC)",
    "scope": (
        "Estimates whether a Model Outcome is possible within the modeled UCI auction. "
        "Not fraud detection, authenticity checking, fairness assessment, or legal validity."
    ),
    "feature_names": FEATURES,
    "numeric_features": NUMERIC_FEATURES,
    "categorical_features": CATEGORICAL_FEATURES,
    "excluded_columns": [EXCLUDED],
    "target": TARGET,
    "class_meanings": {
        "0": "infeasible: verification.result = False (outcome not possible within the modeled auction)",
        "1": "feasible: verification.result = True (outcome possible within the modeled auction)",
    },
    "input_domains": input_domains,
    "split": {
        "method": "stratified train_test_split",
        "test_size": TEST_SIZE,
        "random_state": RANDOM_STATE,
        "train_records": int(len(y_train)),
        "test_records": int(len(y_test)),
    },
    "svm_config": SVM_CONFIG,
    "model_selection": model_selection_report,
    "sklearn_version": sklearn.__version__,
    "python_version": platform.python_version(),
    "dataset": {
        "name": "UCI Auction Verification",
        "url": DATASET_URL,
        "member": DATASET_MEMBER,
        "records": int(len(df)),
        "sha256": DATASET_SHA256,
        "citation": (
            "Ordoni, E., Bach, J., Fleck, A., & Bach, J. (2022). Auction Verification [Dataset]. "
            "UCI Machine Learning Repository. https://doi.org/10.24432/C52K6N"
        ),
        "license": "CC BY 4.0",
    },
    "model_artifact": {"file": MODEL_PATH.name, "sha256": MODEL_SHA256},
    "metrics": metrics,
}

METADATA_PATH.write_text(json.dumps(metadata, indent=2))
METRICS_PATH.write_text(json.dumps(metrics_report, indent=2))

bundle = [MODEL_PATH, METADATA_PATH, METRICS_PATH, CONFUSION_PNG]
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in bundle:
        zf.write(path, arcname=path.name)

# Round-trip check: the exported pipeline reproduces the in-memory predictions.
reloaded = joblib.load(MODEL_PATH)
assert np.array_equal(reloaded.predict(X_test), y_pred), "Reloaded model predictions differ"

with zipfile.ZipFile(ZIP_PATH) as zf:
    print("Bundle contents:", zf.namelist())
print("Model SHA-256  :", MODEL_SHA256)
print("✓ Reloaded pipeline reproduces held-out predictions")


In [ ]:
from google.colab import files

files.download(str(ZIP_PATH))

## 13. Summary: what this notebook demonstrates

**Supervised learning**
* Labelled records: each Auction Configuration carries the model checker's `verification.result`.
* Inputs and target are separated into `X` (seven fields) and `y` (feasible / infeasible); `verification.time` is excluded.
* A fixed stratified 80/20 split reserves an untouched final test partition.

**Support Vector Machine**
* A bounded five-fold search selects `C` and `gamma` by MCC using only the training partition.
* Five-fold validation repeated five times reports the selected configuration's mean and standard deviation.
* The selected class-balanced RBF SVM is evaluated once on the untouched test partition.
* The exported pipeline retains leakage-safe preprocessing and the selected SVM configuration.

**Limits.** Repeated validation measures the selected configuration's stability but is not a fully nested estimate of the selection procedure. All reported results remain specific to the modeled UCI auction and are not formal verification or an assessment of a real-world auction.
